# ハンズオン② DPO — preference data から応答スタイルを学習する

**対応セクション**: 2-2「DPO (Direct Preference Optimization)」  
**推奨所要時間**: 約 60 分  
**必要環境**: GPU (A100 推奨)、SFT 学習済みモデル（またはベースモデル）

---

## このノートブックの目標

1. chosen / rejected ペアの preference データセットを理解する
2. DPOTrainer で SFT モデルに preference alignment を適用する
3. SFT モデルと DPO モデルの出力を同じプロンプトで比較する
4. beta パラメータが応答に与える影響を確認する

## 0. セットアップ

In [ ]:
# 実行時間: 数秒
import os
import json
import torch

os.environ.setdefault('HF_HOME', '/data/shared/hf_cache')
os.environ.setdefault('TRANSFORMERS_CACHE', '/data/shared/hf_cache')
os.environ.setdefault('WANDB_PROJECT', 'dgx-handson-dpo')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'デバイス: {device}')

---

## Step 1: Preference データセットの作成

DPO に必要なのは **(prompt, chosen, rejected)** の3つ組です。  
まず提供されているサンプルデータを確認し、後で自分でも追加してみましょう。

In [ ]:
# 実行時間: 数秒
from datasets import Dataset

# サンプル preference データ
preference_data = [
    {
        'prompt': 'Python でリストを逆順にする方法を教えてください。',
        'chosen': (
            'Python でリストを逆順にするには主に 2 つの方法があります。\n\n'
            '**1. `reversed()` 関数を使う**\n'
            '```python\nmy_list = [1, 2, 3, 4, 5]\n'
            'reversed_list = list(reversed(my_list))\n'
            'print(reversed_list)  # [5, 4, 3, 2, 1]\n```\n\n'
            '**2. スライスを使う**\n```python\nreversed_list = my_list[::-1]\n```\n\n'
            '元のリストを変更せず新しいリストを返します。'
            '元のリスト自体を変更したい場合は `my_list.reverse()` を使います。'
        ),
        'rejected': 'reverse を使います。my_list.reverse() と書けばなります。',
    },
    {
        'prompt': '機械学習における過学習（overfitting）とは何ですか？',
        'chosen': (
            '過学習とは、モデルが訓練データに過度に適合し、未知のデータに対してうまく予測できなくなる現象です。\n\n'
            '**原因**\n- モデルが複雑すぎる\n- 訓練データが少ない\n\n'
            '**対策**\n- 正則化（L1/L2）\n- ドロップアウト\n- データ拡張\n- 早期終了（Early Stopping）\n\n'
            '損失曲線で「訓練損失は下がるがバリデーション損失が上がる」パターンが典型的なサインです。'
        ),
        'rejected': '訓練データを覚えすぎることです。たくさんデータを集めれば解決します。',
    },
    {
        'prompt': 'Git の rebase と merge の違いを教えてください。',
        'chosen': (
            'どちらもブランチの変更を統合する操作ですが、履歴の形が異なります。\n\n'
            '**merge**: ブランチが分岐したままマージコミットが作られます。誰がいつマージしたかが明確です。\n\n'
            '**rebase**: コミットが積み直されて履歴が一直線になります。すっきりしますが、コミット ID が変わります。\n\n'
            'チーム開発では merge が安全。個人ブランチの整理には rebase が有効です。'
        ),
        'rejected': 'rebase はコミットを移動させます。merge はブランチをくっつけます。',
    },
]

dataset = Dataset.from_list(preference_data)
print(f'データ件数: {len(dataset)}')
print('\n--- 1件目のサンプル ---')
print(f'prompt  : {dataset[0]["prompt"]}')
print(f'chosen  : {dataset[0]["chosen"][:100]}...')
print(f'rejected: {dataset[0]["rejected"]}')

---

## Step 2: モデルと LoRA の設定

In [ ]:
# 実行時間: 約2〜3分
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model

# SFT 済みモデルがあればそのパスに変更
BASE_MODEL = 'meta-llama/Meta-Llama-3-8B'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
) if device == 'cuda' else None

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto' if device == 'cuda' else None,
    torch_dtype=torch.bfloat16 if device == 'cuda' else torch.float32,
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    bias='none',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

---

## Step 3: DPO 学習の実行

`beta` が DPO の KL 正則化係数です。大きいほど参照モデルから逸脱しにくくなります。

In [ ]:
# 実行時間: 約5〜10分（max_steps=20 の場合）
from trl import DPOTrainer, DPOConfig
from datetime import datetime

# ── ここを変えて実験しよう ──────────────────────────────────
BETA = 0.1    # 試す値: 0.05, 0.1, 0.3
# ──────────────────────────────────────────────────────────

output_dir = f'./outputs/dpo_{datetime.now().strftime("%H%M%S")}'

dpo_args = DPOConfig(
    output_dir=output_dir,
    max_steps=20,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    beta=BETA,
    bf16=device == 'cuda',
    logging_steps=5,
    report_to='wandb',
    run_name=f'dpo-beta{BETA}-demo',
    max_length=512,
    max_prompt_length=256,
)

trainer = DPOTrainer(
    model=model,
    args=dpo_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

print('DPO 学習開始...')
trainer.train()
print('学習完了！')

---

## Step 4: SFT モデルと DPO モデルの出力比較

同じプロンプトに対して、DPO 前後の出力がどう変わるか確認します。

In [ ]:
# 実行時間: 約30秒
model.eval()

test_prompts = [
    'Python でリストをソートする方法を 3 つ教えてください。',
    'REST API と GraphQL の違いを説明してください。',
]

for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(
        output_ids[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True,
    )
    print(f'\n=== プロンプト ===')
    print(prompt)
    print(f'=== DPO 後の回答 (beta={BETA}) ===')
    print(generated)
    print('-' * 40)

---

## まとめ

1. **Preference データ** は (prompt, chosen, rejected) の3つ組で、chosen と rejected の差が明確であるほど効果的
2. **DPO** は報酬モデル不要・PPO 不要で、通常の SFT と同じ枠組みで実装できる
3. **beta** が大きいほど参照モデルに近い保守的な出力になる
4. 少量のデータ（数件〜数十件）でも応答スタイルの変化を確認できる

次のハンズオンでは、LangChain + ChromaDB で RAG パイプラインを構築します。